# Sesión 4 · Ejercicio 1 — Adaptar sin reentrenar
**Objetivo:** tomar un modelo de difusión preentrenado que NO conoce
una clase, y enseñársela tocando ~5 % de sus parámetros
(adaptación de bajo rango, LoRA).
**Tiempo:** MÍNIMO 25 min · COMPLETO 45 min
**Produce:** §7 de su bitácora — comparación antes/después + nota de
qué se logró controlar y qué no
**Ruta B (tabular/series):** al final del cuaderno.


### Cómo trabajar este cuaderno (1 minuto de lectura)

1. **Guarde su copia**: Archivo → Guardar una copia en Drive. Si no, pierde su trabajo al cerrar.
2. Ejecute las celdas **en orden**. Solo las marcadas `#### OBLIGATORIO ####` producen su entregable; las de **EXTENSIÓN** son opcionales, para quien le sobre tiempo.
3. ¿Algo no corre, o tarda demasiado? Ejecute la **CELDA DE RESCATE**: carga resultados ya calculados y usted sigue con el análisis. Usarla **no descuenta puntos** — solo dígalo en su bitácora.
4. Al terminar, copie la figura y sus observaciones (2-3 líneas con sus palabras) a la sección de su **bitácora** que dice el encabezado. Eso es TODO el entregable — no se pide nada más.


In [ ]:
#### OBLIGATORIO #### — setup (idempotente: puede ejecutarla dos veces)
import os, sys
if not os.path.isdir("src"):
    if not os.path.isdir("IAA6_M13_Gen"):
        !git clone -q https://github.com/AdriannaGmz/IAA6_M13_Gen
    %cd IAA6_M13_Gen
# Si ya había una copia en esta máquina, se pone al día: de lo
# contrario se quedaría con la versión del día que la clonó, y las
# correcciones publicadas después nunca le llegarían.
!git pull -q --ff-only
!pip install -q -r requirements.txt
sys.path.insert(0, ".")
from src import datos, modelos, evaluar, graficas, rescate
MODO_GPU = rescate.hay_gpu()   # imprime "GPU disponible" o "Modo CPU"


### Paso 1 · El escenario: un proveedor que hace casi todo

Éste es el escenario de su vida profesional: llega un modelo
preentrenado que hace casi todo bien… **menos lo que a usted le
urge**. El difusor que va a cargar se entrenó con todas las clases
del conjunto de demostración **excepto** la minoritaria (`anillo`):
esa clase simplemente no existe en sus pesos.

Al cargar el checkpoint pueden aparecer una o dos figuras «gratis»
(precomputadas, guardadas dentro del archivo): ignórelas — las suyas
vienen en las celdas siguientes.


In [ ]:
#### OBLIGATORIO #### — el modelo preentrenado (al que le falta algo)
# Este modelo se entrenó con TODAS las clases del conjunto de
# demostración MENOS la minoritaria («anillo»). Nunca la vio.
X, y, meta = datos.cargar("imagen")
contenido, _ = rescate.cargar("s4_lora")
difusion = modelos.Difusion(meta, pasos=contenido["pasos"])
difusion.red.load_state_dict(contenido["state_dict_base"])
CLASE = meta["nombres_clases"].index(meta["clase_minoritaria"])
print(f"Clase objetivo: {meta['clase_minoritaria']!r} (índice {CLASE})")


### Paso 2 · Documentar el fracaso (su «antes»)

Antes de arreglar nada, pídale al modelo justo lo que no sabe hacer.
Lo esperable: **ni un anillo** — cuadros ruidosos con fragmentos de
las clases que sí conoce. Esa figura es su punto de comparación; sin
un «antes» documentado, el «después» no demuestra nada.


In [ ]:
#### OBLIGATORIO #### — ANTES de adaptar: pídale lo que no sabe hacer
antes = difusion.muestrear(8, y=CLASE, pasos=50, guia=3.0)
graficas.rejilla(antes, "ANTES: pide 'anillo' a un modelo que nunca "
                        "vio anillos", n=8)


### Paso 3 · El adaptador — accesorio, no cirugía

LoRA en dos renglones: se **congelan** todos los pesos del modelo y
junto a cada capa elegida se añade un «desvío» de dos matrices flacas
(rango `r`); **sólo el desvío se entrena**. Es ponerle un accesorio
al modelo en lugar de operarlo — y el accesorio es un archivo aparte
que se pone y se quita.

Qué esperar de la salida: unos **73 mil parámetros entrenables de
1.4 millones — alrededor del 5 %**. Ese número va a su §7.


In [ ]:
#### OBLIGATORIO #### — el adaptador de bajo rango (LoRA)
# LoRA (Hu et al., ICLR 2022): junto a cada capa elegida se añade un
# desvío de rango r; SÓLO eso se entrena. El modelo base queda intacto.
# COMPLETAR: el rango r y las capas objetivo del adaptador.
# Pista 1: r = 8 es un punto de partida razonable (y lora_alpha = 2r).
# Pista 2: las capas adaptables ya están listadas en CAPAS (todas las
#          convoluciones y lineales de la red; peft no adapta las
#          transpuestas del decodificador).
from peft import LoraConfig, get_peft_model
CAPAS = ["baja1.0", "baja2.0", "baja3.0", "baja4.0", "medio.0",
         "medio.2", "salida", "p2", "p3", "p4"]
config = LoraConfig(
    r=..., lora_alpha=...,
    target_modules=...,
    modules_to_save=["emb"],   # el embebido de clase sí se reentrena
)
difusion.red = get_peft_model(difusion.red, config)
difusion.red.print_trainable_parameters()


### Paso 4 · Adaptar con lo poco que hay

Con ~160 anillos reales no se entrena un difusor desde cero — pero
para un desvío del 5 % **alcanza**. Dos minutos de CPU.

Qué esperar: la pérdida casi no baja (≈0.12 → 0.10) e incluso repunta
al final. **Es normal**: el modelo base ya sabía quitar ruido; el
adaptador sólo lo redirige. No es señal de fracaso.


In [ ]:
#### OBLIGATORIO #### — adaptar con los pocos ejemplos que hay (~2 min)
# COMPLETAR: seleccione SOLAMENTE los ejemplos reales de la clase
# objetivo para adaptar.
# Pista: una máscara booleana sobre y, como en S2-E2.
mascara = ...
print(f"Ejemplos de adaptación: {int(mascara.sum())}")
historial = difusion.entrenar(
    X[mascara], y[mascara], epocas=100,
    cb=lambda e, r: (e % 20 == 19) and print(
        f"  época {e + 1:3d}/100 · pérdida {r['perdida']:.3f}"))


In [ ]:
# ── CELDA DE RESCATE ────────────────────────────────────────
# ¿No corrió la adaptación? Esto carga el adaptador ya entrenado.
from peft import LoraConfig, get_peft_model
contenido, figuras = rescate.cargar("s4_lora")
difusion = modelos.Difusion(meta, pasos=contenido["pasos"])
difusion.red.load_state_dict(contenido["state_dict_base"])
config = LoraConfig(r=8, lora_alpha=16,
                    target_modules=contenido["config"]["capas"],
                    modules_to_save=["emb"])
difusion.red = get_peft_model(difusion.red, config)
difusion.red.load_state_dict(contenido["state_dict_adaptado"])
print("Adaptador precomputado cargado.")


### Paso 5 · El «después» — y qué exigirle

La misma petición de antes, al modelo ya adaptado. No espere los
anillos del difusor de ayer: espere anillos **toscos, irregulares,
pero inconfundibles** — masas con hueco central sobre fondo limpio.
Control aproximado con 160 ejemplos y el 5 % de los parámetros: eso
es LoRA — redirección barata, no milagros.


In [ ]:
#### OBLIGATORIO #### — DESPUÉS: la misma petición, lado a lado
despues = difusion.muestrear(8, y=CLASE, pasos=50, guia=3.0)
fig = graficas.comparar(antes, despues,
                        etiquetas=("antes de adaptar", "después (LoRA)"))
fig.savefig("bitacora_s4e1_antes_despues.png", dpi=120)


### Paso 6 · La otra cara: ¿qué se rompió?

El paso que separa al profesional del aficionado: verificar qué se
**degradó** de lo que el modelo ya sabía. Va a pedirle una clase que
dominaba antes de la adaptación. Si sale dañada (suele pasar con la
más parecida al anillo), acaba de fotografiar el **costo oculto de
adaptar** — el hallazgo más citable de su §7.


In [ ]:
#### OBLIGATORIO #### — ¿y qué NO se logró controlar?
# La otra cara: ¿el adaptador estropeó lo que el modelo YA sabía?
OTRA = 0 if CLASE != 0 else 1   # una clase mayoritaria cualquiera
graficas.rejilla(difusion.muestrear(8, y=OTRA, pasos=50, guia=3.0),
                 f"Después de adaptar: ¿aún sabe hacer "
                 f"{meta['nombres_clases'][OTRA]!r}?", n=8)


### Observación — ¿qué logré controlar y qué no?

- ¿El modelo adaptado produce anillos reconocibles? ___
- ¿Cuántos parámetros se entrenaron, y qué fracción del total es? ___
- ¿La clase que ya sabía se degradó al adaptar? ___
- Si su respuesta anterior fue «sí, algo»: acaba de observar el costo
  oculto de adaptar. Anótelo, es el hallazgo más citable de su §7.


In [ ]:
#### OBLIGATORIO #### — artefacto para la bitácora
print("Copie este bloque en la sección §7 de su bitácora y adjunte")
print("bitacora_s4e1_antes_despues.png:\n")
print(f"- Clase enseñada: {meta['clase_minoritaria']!r} · rango r = 8")
print("- Parámetros entrenados: <n> de <total> (<porcentaje>)")
print("- ¿Qué logré controlar?: <...>")
print("- ¿Qué no?: <...>")


### Ruta B (proyectos tabulares o de series)
Si su proyecto no es de imágenes, el equivalente de «adaptar sin
reentrenar» es refinar el CVAE condicional de S2-E2 con datos
ampliados y variar la condición: mismo esqueleto (antes → adaptar →
después → qué controlé y qué no), mismo artefacto para §7.

```python
# punto de partida — refinar SOLO 10 épocas sobre datos nuevos:
# cvae.entrenar(X_nuevos, y_nuevos, epocas=10)
# y comparar muestrear(y=condicion) antes y después.
```


### EXTENSIÓN (equipos rápidos)
Repita la adaptación con `r=1` y con `r=32`. ¿El rango 1 alcanza para
aprender la clase? ¿El 32 mejora algo visible o sólo gasta parámetros?
(El control ESTRUCTURAL —imponer bordes o poses con una red de control
tipo ControlNet, Zhang et al., ICCV 2023— pide modelos más grandes que
los de esta práctica; queda como lectura del seminario.)
